# 02 - Prepare Dataset & Build English Token Vocabulary

This notebook inspects the continuous ISL CSV annotations, demonstrates standardized text tokenization, builds the target word-token vocabulary (with CTC `<blank>` reserved at ID 0), loads keypoint sample arrays, and validates the CTC sequence length constraint ($T \ge L$).

> **Dataset Note:** The available continuous ISL annotations are English transcript tokens used as sequence targets for CTC training. They are not manually annotated ISL glosses.

In [ ]:
# 1. Import modules & configuration
import os, sys, csv, json
from pathlib import Path
import numpy as np

WORKING_DIR = Path("/kaggle/working/sentence-model-training") if os.path.exists("/kaggle") else Path("..")
sys.path.insert(0, str(WORKING_DIR.resolve()))

from config import (
    CSV_UID_COLUMN,
    CSV_TEXT_COLUMN,
    MIN_TOKEN_FREQ,
    KEYPOINT_FILENAME_PATTERN,
    VOCABULARY_FILE,
)
from src.vocabulary import GlossVocabulary, WordVocabulary, tokenize_english_transcript
from src.dataset import SentenceISLDataset, check_ctc_validity
print("Modules and configuration imported successfully.")

In [ ]:
# 2. Inspect CSV Annotation Columns & Sample Rows
CSV_ANNOTATIONS_PATH = Path("/kaggle/input/isltranslate/annotations.csv") if os.path.exists("/kaggle") else Path("../outputs/sample_annotations.csv")

# If file doesn't exist locally, create a small sample demonstration CSV
if not CSV_ANNOTATIONS_PATH.exists():
    CSV_ANNOTATIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
    sample_rows = [
        {"uid": "sample_001", "english": "GOOD MORNING", "split": "train"},
        {"uid": "sample_002", "english": "HOW ARE YOU?", "split": "train"},
        {"uid": "sample_003", "english": "THANK YOU!", "split": "train"},
        {"uid": "sample_004", "english": "WHAT IS YOUR NAME", "split": "train"},
        {"uid": "sample_005", "english": "NICE TO MEET YOU.", "split": "val"},
        {"uid": "sample_006", "english": "PLEASE HELP ME", "split": "val"},
    ]
    with open(CSV_ANNOTATIONS_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["uid", "english", "split"])
        writer.writeheader()
        writer.writerows(sample_rows)
    print(f"Created demonstration CSV at: {CSV_ANNOTATIONS_PATH}")

with open(CSV_ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    print(f"CSV Columns: {reader.fieldnames}")
    print("\n--- First 5 Sample Rows ---")
    for i, row in enumerate(reader):
        if i >= 5:
            break
        print(f"Row {i+1}: UID={row.get(CSV_UID_COLUMN, row.get('uid'))} | Transcript='{row.get(CSV_TEXT_COLUMN, row.get('english'))}'")

In [ ]:
# 3. Build & Save Vocabulary from English Transcripts
vocab = GlossVocabulary()
vocab.build_from_csv(
    csv_path=CSV_ANNOTATIONS_PATH,
    text_col=CSV_TEXT_COLUMN if CSV_TEXT_COLUMN in reader.fieldnames else "english",
    min_freq=MIN_TOKEN_FREQ,
)

vocab_out_path = Path("/kaggle/working/outputs/gloss_vocab.json") if os.path.exists("/kaggle") else VOCABULARY_FILE
vocab.save_to_json(vocab_out_path)

print(f"Vocabulary built with {len(vocab)} tokens (including Blank=0, Unk=1).")
print(f"Top tokens: {list(vocab.token_to_id.items())[:10]}")

In [ ]:
# 4. Demonstrate Text Normalization & Token-to-ID Conversion
sample_phrase = "How are you, today?"
tokens = tokenize_english_transcript(sample_phrase)
token_ids = vocab.encode_tokens(tokens)
decoded_tokens = vocab.decode_tokens(token_ids)

print(f"Original text:     '{sample_phrase}'")
print(f"Normalized tokens: {tokens}")
print(f"Token IDs:         {token_ids}")
print(f"Decoded tokens:    {decoded_tokens}")

In [ ]:
# 5. Load One Keypoint Sample & Verify Shapes and CTC Validity
KEYPOINTS_DIR = Path("/kaggle/working/keypoints_cache") if os.path.exists("/kaggle") else Path("../keypoints_cache")
KEYPOINTS_DIR.mkdir(parents=True, exist_ok=True)

# Create sample keypoint file if not present for demonstration
demo_kp_file = KEYPOINTS_DIR / "sample_001.npy"
if not demo_kp_file.exists():
    # T=36 frames, 543 landmarks, 3 coordinates
    dummy_kp = np.random.randn(36, 543, 3).astype(np.float32) * 0.1
    np.save(demo_kp_file, dummy_kp)

# Instantiate dataset on small sample
dataset = SentenceISLDataset(
    csv_path=CSV_ANNOTATIONS_PATH,
    keypoints_dir=KEYPOINTS_DIR,
    vocab=vocab,
    uid_col="uid",
    text_col="english",
    keypoint_filename_pattern="{uid}.npy",
)

sample = dataset[0]
kp_tensor = sample["keypoints"]
target_tensor = sample["target"]
t_len = sample["input_len"]
l_len = sample["target_len"]

print(f"Sample UID:            {sample['sample_id']}")
print(f"Keypoints shape:       {kp_tensor.shape} (T, V=543, C=3)")
print(f"Target token IDs:      {target_tensor.tolist()} (Length={l_len})")
print(f"Original Transcript:   '{sample['text']}'")

# Check CTC Length Validity: T >= L
is_valid_ctc = check_ctc_validity(input_length=t_len, target_length=l_len, sample_id=sample['sample_id'])
print(f"CTC Validity Check:    {'PASSED (T >= L)' if is_valid_ctc else 'FAILED (T < L)'} [{t_len} >= {l_len}]")